# HalluDet-Lite Phase 5 – Phi-2 DPO Fine-Tuning (Kaggle)

**Run this notebook in Kaggle with GPU accelerator enabled.**

**Settings (right panel) > Accelerator > GPU T4 x2 (or T4 / P100)**

## ⚠️ IMPORTANT – Run cells in order, restart kernel after Cell 1
After Cell 1 installs packages, go to **Run > Restart & Run All** (or restart kernel + re-run).
This is required on Kaggle so the freshly installed versions are actually loaded.

## What this does
Fine-tunes Microsoft Phi-2 (2.7B params) using Direct Preference Optimization (DPO)
on hallucination preference pairs from HaluEval.

The fine-tuned LoRA adapter (~40MB) is saved to `/kaggle/working/halludet_phi2/`.
After the run, download it from the **Output** tab on the right panel.

## Hardware
- T4 GPU (15GB VRAM) or P100 (16GB VRAM) – Kaggle free tier
- Session time: ~2 hours total
- Checkpoints every 50 steps – safe to resume if session expires

## Dataset
Two options (choose one in Cell 4):
- **Option A (default)**: Build the dataset automatically from HaluEval (no upload needed, ~5 min)
- **Option B**: Upload `dpo_dataset.json` as a Kaggle Dataset and attach it to this notebook

In [ ]:
# ── Cell 1: Install / upgrade dependencies ────────────────────────────────────
# Use latest compatible versions – do NOT pin old versions (causes API mismatches)
import subprocess, sys

pkgs = [
    'transformers>=4.46.0',   # eval_strategy (not evaluation_strategy)
    'datasets',
    'peft>=0.12.0',           # prepare_model_for_kbit_training improvements
    'trl>=0.12.0',            # processing_class (not tokenizer), DPOConfig fixes
    'bitsandbytes>=0.43.0',   # NF4 on Kaggle T4/P100 GPU
    'accelerate>=0.30.0',
    'huggingface_hub',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs)
print('✅ Dependencies installed.')
print()
print('⚠️  NEXT: Go to Run > Restart & Run All to load the new library versions.')
print('   (Skip this if you already restarted after installing.)')

In [ ]:
# ── Cell 2: Verify GPU & set output directory ─────────────────────────────────
import torch
import os

# /kaggle/working/ persists after the run and files are downloadable from Output tab
OUTPUT_DIR = '/kaggle/working/halludet_phi2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError(
        'No GPU detected!\n'
        'Go to the right panel > Settings > Accelerator > GPU T4, then restart the kernel.'
    )

# Print installed versions for debugging
import transformers, peft, trl, bitsandbytes
print(f'   transformers: {transformers.__version__}')
print(f'   peft:         {peft.__version__}')
print(f'   trl:          {trl.__version__}')
print(f'   bitsandbytes: {bitsandbytes.__version__}')
print(f'   Output dir:   {OUTPUT_DIR}')

In [ ]:
# ── Cell 3: Hugging Face login (optional – Phi-2 is public) ──────────────────
# If you have a HF token, add it as a Kaggle Secret named 'HF_TOKEN':
#   Notebook > Add-ons > Secrets > Add New Secret
# Get your token from: https://huggingface.co/settings/tokens

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Logged in to Hugging Face via Kaggle secret.')
except Exception as e:
    print(f'[INFO] No HF_TOKEN secret found ({e}).')
    print('       Phi-2 is a public model – downloading without login.')
    print('       If you hit a 401 error, add HF_TOKEN as a Kaggle secret.')

In [ ]:
# ── Cell 4: Load dataset ──────────────────────────────────────────────────────
#
# OPTION A (DEFAULT) – Build from HaluEval automatically ──────────────────────
#   USE_HALUEVAL = True   ← keep this (requires internet, ~5 min)
#
# OPTION B – Use your own uploaded dpo_dataset.json ───────────────────────────
#   1. Click '+ Add Data' in the right panel
#   2. Upload > select data/processed/dpo_dataset.json
#   3. Set USE_HALUEVAL = False  and update DATA_PATH below
#      e.g. DATA_PATH = '/kaggle/input/halludet-dpo/dpo_dataset.json'
# ─────────────────────────────────────────────────────────────────────────────

USE_HALUEVAL = True
DATA_PATH    = '/kaggle/input/halludet-dpo/dpo_dataset.json'   # only if USE_HALUEVAL=False

import json, random
from datasets import Dataset, load_dataset

if USE_HALUEVAL:
    print('Building DPO dataset from pminervini/HaluEval ...')
    halueval = load_dataset('pminervini/HaluEval', 'qa_samples', split='train')
    pairs = [
        {
            'prompt':   row['question'],
            'chosen':   row['right_answer'],
            'rejected': row['hallucinated_answer']
        }
        for row in halueval
    ]
    random.seed(42)
    random.shuffle(pairs)
    n_val    = max(1, int(len(pairs) * 0.1))
    train_ds = Dataset.from_list(pairs[n_val:])
    eval_ds  = Dataset.from_list(pairs[:n_val])
    print(f'✅ HaluEval loaded. Train: {len(train_ds):,} | Val: {len(eval_ds):,}')
else:
    print(f'Loading dataset from: {DATA_PATH}')
    with open(DATA_PATH) as f:
        raw = json.load(f)
    train_ds = Dataset.from_list(raw['train'])
    eval_ds  = Dataset.from_list(raw['val'])
    print(f'✅ Loaded. Train: {len(train_ds):,} | Val: {len(eval_ds):,}')

# Inspect one sample
print('\nSample row:')
print('  prompt:  ', train_ds[0]['prompt'][:80])
print('  chosen:  ', train_ds[0]['chosen'][:80])
print('  rejected:', train_ds[0]['rejected'][:80])

In [ ]:
# ── Cell 5: Load Phi-2 with QLoRA ─────────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL = 'microsoft/phi-2'

# 4-bit NF4 quantisation — works on T4 and P100 (fp16 only, NOT bf16)
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

print('Loading Phi-2 (this takes ~3-5 min on first download)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.use_cache = False  # required for gradient checkpointing

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'   # DPO requires left-padding for batch decoding

# FIX: prepare for kbit training BEFORE applying LoRA
# This casts LayerNorms to fp32 and enables input gradients (required with gradient_checkpointing)
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

# FIX: include ALL Phi-2 linear layers (attention + MLP) for best coverage
# Phi-2 attention: q_proj, k_proj, v_proj, dense
# Phi-2 MLP:       fc1, fc2
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'dense', 'fc1', 'fc2'],
    bias='none',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print('✅ Phi-2 + QLoRA loaded.')

In [ ]:
# ── Cell 6: Load reference model (frozen base for DPO KL term) ───────────────
# The reference model must NOT have LoRA – it stays frozen throughout training.
print('Loading frozen reference Phi-2 ...')
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
# Disable gradients for reference model — nothing should be trained
for param in ref_model.parameters():
    param.requires_grad = False

print('✅ Reference model loaded (frozen).')

In [ ]:
# ── Cell 7: DPO Training ───────────────────────────────────────────────────────
import glob
from trl import DPOTrainer, DPOConfig

# Resume from checkpoint if a previous run was interrupted
checkpoints = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))
resume_from = checkpoints[-1] if checkpoints else None
if resume_from:
    print(f'Resuming from checkpoint: {resume_from}')
else:
    print('Starting fresh training run.')

# FIX 1: evaluation_strategy → eval_strategy (changed in transformers ≥ 4.46)
# FIX 2: max_length / max_prompt_length now live inside DPOConfig (not TrainingArguments)
# FIX 3: optim 'paged_adamw_8bit' requires bitsandbytes ≥ 0.43 (already installed)
cfg = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=0.1,
    num_train_epochs=2,
    per_device_train_batch_size=2,          # reduced to fit 15 GB VRAM comfortably
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,          # effective batch = 16
    learning_rate=5e-5,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    fp16=True,
    bf16=False,                             # T4 / P100 do NOT support bf16
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    max_length=512,
    max_prompt_length=256,
    eval_strategy='steps',                  # FIX: was 'evaluation_strategy' (removed ≥4.46)
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    logging_steps=10,
    report_to='none',                       # set to 'wandb' if you have a W&B account
    remove_unused_columns=False,
    dataloader_num_workers=0,               # safest for Kaggle multi-process issues
)

# FIX: tokenizer= is deprecated in trl ≥ 0.12 → use processing_class=
trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=cfg,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,             # FIX: was 'tokenizer=' (deprecated)
)

print('🚀 Starting DPO training...')
print('   Watch: rewards/chosen should increase, rewards/rejected should decrease')
print('   If kl_divergence > 5.0, increase beta (currently 0.1) in DPOConfig')
trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# ── Cell 8: Save final adapter ────────────────────────────────────────────────
final_dir = f'{OUTPUT_DIR}/final'
os.makedirs(final_dir, exist_ok=True)

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

print(f'✅ LoRA adapter saved to: {final_dir}')
print()
print('NEXT STEPS (Kaggle):')
print('1. Wait for the notebook run to finish')
print('2. Click the "Output" tab on the right panel')
print('3. Download  halludet_phi2/final/  from  /kaggle/working/')
print('4. Place it on your PC at:  halludet_lite/models/phi2_dpo_lora/final/')
print('5. Run:  python scripts/run_all_eval.py')

In [ ]:
# ── Cell 9 (Optional): Quick sanity test on the fine-tuned model ──────────────
print('Testing fine-tuned model on sample questions...')

test_questions = [
    'When did Albert Einstein win the Nobel Prize?',
    'What is the capital of Australia?',
    'Who wrote Romeo and Juliet?',
]

model.eval()
for q in test_questions:
    prompt = f'Answer accurately: {q}\nAnswer:'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    answer     = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print(f'Q: {q}')
    print(f'A: {answer.strip()}')
    print()